# 07 — GPU Runtime & Dual-Coder Server Benchmarking (`Agent 07`)

- **Active Run ID**: `run_20261111_002`
- **Owner**: Agent 07 (`src/runtime/`, `src/model_runtime.py`)
- **Canonical Output**: `runs/run_20261111_002/gpu_runtime_config.json` (`gpu_runtime_config_v1`)

## Mission & Architectural Invariants
1. **Runtime Hardware Detection**: Detect physical accelerator and available VRAM at runtime (never assume hardware from documentation).
2. **Model-Family Diversity**:
   - **Coder A**: DeepSeek reasoning-oriented 32B quantized model (`deepseek-ai/DeepSeek-R1-Distill-Qwen-32B-AWQ`).
   - **Coder B**: Independent multilingual model family (`mistralai/Mistral-Small-3.1-24B-Instruct-2503-AWQ` preferred; `google/gemma-3-27b-it-qat-q4_0-gguf` alternative). Qwen-derived substitutes for Coder B are strictly blocked.
3. **5-Stage Runtime Benchmark & Empirically Derived Memory Budgets**:
   - Stage 1: Coder A alone
   - Stage 2: Coder B alone
   - Stage 3: Both servers resident
   - Stage 4: One simultaneous structured request to each
   - Stage 5: Bounded concurrent requests & failure recovery
4. **Crash-Safe Atomic Question Checkpoints & Sequential Fallback**:
   - Every accepted response records `run_id`, `question_id`, `coder_id`, `prompt_hash`, `source_hash`, `model_repo`, `model_revision`, `quantization`, `generation_config`, `raw_response`, `parsed_response`, `schema_status`, `runtime_sec`, and `peak_vram_mb`.


In [ ]:
from pathlib import Path
import json
from src.runtime import (
    detect_hardware_profile,
    build_and_save_runtime_config,
    validate_model_family_diversity,
    DEFAULT_CODER_A_SPEC,
    DEFAULT_CODER_B_MISTRAL_SPEC,
    ALTERNATIVE_CODER_B_GEMMA_SPEC,
)

hw = detect_hardware_profile()
print(json.dumps(hw.to_dict(), indent=2))


In [ ]:
# Validate model-family diversity (Coder A DeepSeek 32B vs Coder B Mistral Small 3.1 24B / Gemma 3 27B)
validate_model_family_diversity(DEFAULT_CODER_A_SPEC, DEFAULT_CODER_B_MISTRAL_SPEC)
validate_model_family_diversity(DEFAULT_CODER_A_SPEC, ALTERNATIVE_CODER_B_GEMMA_SPEC)
print('Model-family diversity verified successfully.')


In [ ]:
# Execute the 5-stage runtime benchmark and derive memory budgets from measurements
cfg, bench_report, cfg_path = build_and_save_runtime_config(run_id='run_20261111_002')
print(f'Saved config to: {cfg_path}')
print(json.dumps(bench_report['derived_memory_budget'], indent=2))
